# Segment Specimens

Finds every beetle in every photo of a dataset and cuts each one out along its outline, clip-art style.

1. **Grounding DINO** looks for "beetle" in each photo and draws a box around every one it finds.
   - **Look-alikes** (e.g. a bee or wasp): CLIP, an image classifier, scores each box. There are two ways to turn its scores into a decision (`SCORING_MODE` in Step 1): **joint**, one question over every category at once (beetle, bee, wasp... or neither), and **separate**, one yes/no question per category (is there an insect? is it a beetle? is it a bee? a wasp?...). Boxes that aren't beetles are sorted into folders you can check. In separate mode, a box that scores high as both a beetle and a look-alike is set aside **for review** instead of being guessed. `Tune_Thresholds.ipynb` compares the two modes on examples you label.
   - **No beetle found** (e.g. a photo of just tape and a label): the photo is set aside in a `no_beetle/` folder for you to review instead of being used.
   - **Several beetles found**: each beetle becomes its own image.
2. **SAM** (Segment Anything) traces the exact outline of the beetle inside each box.
3. Each beetle is cropped to its outline and saved twice:
   - a **training-ready JPG** on a plain background, in the same folder layout as the source, so `Train_Model.ipynb` can use it directly
   - a **transparent PNG** cutout.

**Output**, for a source folder `Data/My_Dataset/`:

```
Data/My_Dataset_segmented/                 training-ready JPGs, same layout as the source
    training/Carabidae/<photo>.jpg           (one beetle in the photo)
    training/Carabidae/<photo>_beetle01.jpg  (several beetles: numbered top-to-bottom)
    ...
Data/My_Dataset_segmented_extras/
    cutouts/...                              transparent PNGs, same layout
    no_beetle/...                            copies of photos where no beetle was found
    rejected/lookalike/<name>/...            boxes rejected as a bee, wasp, etc. (small crops)
    rejected/neither/...  rejected/no_insect/...   boxes that were not a beetle, nor a look-alike / not an insect
    review/...                               separate mode only: boxes that scored as both beetle and look-alike
                                             (training-ready JPG and PNG cutout, ready to copy into the output if they are beetles)
    tuning/...                               scores for choosing the thresholds (Step 3b)
    segmentation_log.csv                     one row per photo: what was found and saved
```

Your original photos are never modified.

**Tip:** run `Split_Dataset.ipynb` *first*, then segment the split dataset. That way every beetle from a multi-beetle photo lands in the same split (training or testing), so the model is never tested on a beetle it was trained on.

**Kernel:** select the **specimen** environment (top-right of the notebook). Besides TensorFlow for the other notebooks, it has PyTorch and Hugging Face `transformers`, which Grounding DINO, SAM and CLIP need. The first run downloads the three models (about 1.6 GB); after that they load from your disk.

**Speed:** on a CPU, expect roughly 10–25 seconds per photo, so about 3–7 hours per 1,000 photos. Step 4 can be stopped and resumed, so it doesn't have to run in one sitting.

**How to use it:** edit Step 1, run Steps 2 and 3 to preview a few photos, optionally run Step 3b and `Tune_Thresholds.ipynb` to compare the two scoring modes and pick their thresholds from labelled examples, then run Step 4 to process everything.

## Step 1: Settings

- **`SOURCE_DIR`**: the folder to process, e.g. `"My_Dataset"` for `Data/My_Dataset/`. Every image in it and in all its subfolders is processed. Leave it as `None` to pick the folder with a pop-up.
- **`OUTPUT_NAME`**: name for the output folder in `Data/`. `None` uses the source folder's name plus `_segmented`.
- **`PROMPT`**: what Grounding DINO looks for. End it with a full stop.
- **`BOX_THRESHOLD`**: how confident Grounding DINO must be to count something as a beetle. **Lower** it if beetles are being missed; **raise** it if tape, labels or scale bars are being picked up as beetles.
- **`TEXT_THRESHOLD`**: how closely a box must match the words in `PROMPT`. Rarely needs changing.
- **`LOOKALIKES`**: insects that get mistaken for beetles, e.g. `["bee", "wasp", "ant", "fly"]`. Each one gets its own score.
- **`SCORING_MODE`**: how CLIP's scores become a decision. Try `"joint"` first; if it loses too many real beetles or lets too many look-alikes through, try `"separate"`. Step 3b scores both at once, so `Tune_Thresholds.ipynb` can compare them without running anything again.

  **Joint mode.** One question over every category: CLIP gives each box a score for "beetle", for each look-alike (the highest is the *look-alike score*), and for "other" (labels, tape...). They are probabilities that add up to 1, so a box can't score high for two things at once. A box is a **look-alike** if its look-alike score is at least `LOOKALIKE_MIN` and beats its beetle score. Otherwise it is a **beetle** if its beetle score is at least `BEETLE_MIN`. Otherwise it is **not an insect**.
  - **`BEETLE_MIN`**: keep it **low** (lenient) so real beetles aren't lost.
  - **`LOOKALIKE_MIN`**: keep it **high** (strict) so only clear look-alikes are rejected.

  **Separate mode.** One yes/no question per category, each scored on its own, so a box *can* score high for two things. The look-alike score is the highest of the individual look-alike scores. First, if the *insect* score is below `SEP_INSECT_MIN`, it is **not an insect**. Then:

  | beetle score ≥ `SEP_BEETLE_MIN` | look-alike score ≥ `SEP_LOOKALIKE_MIN` | decision |
  |---|---|---|
  | yes | no | **beetle** (kept) |
  | yes | yes | **review** (set aside for you to check) |
  | no | yes | **look-alike** (rejected) |
  | no | no | **neither** (not a beetle or a look-alike; rejected) |

  A "beetle" score here means *"this is a beetle, not just any insect"*, and a "bee" score means *"this is a bee, not just any insect"*, which is why a good beetle photo should score low as a bee.
- **`TUNING_SAMPLE_SIZE`**: how many random photos Step 3b scores for threshold tuning.
- **`BACKGROUND_COLOR`**: the colour behind the beetle in the training-ready JPGs, as (red, green, blue). `(255, 255, 255)` is white.
- **`PADDING`**: pixels of background kept around each beetle's outline.
- **`MAKE_SQUARE`**: `Train_Model.ipynb` resizes every image to 224×224, which *stretches* anything that isn't square, so a long, thin beetle would be squashed. With `True`, each training JPG is padded to a square with `BACKGROUND_COLOR` (beetle centred), so that resize only scales the beetle. The images keep their original resolution; `Train_Model.ipynb` does the resizing. `False` keeps the tight crop. The transparent PNG cutouts are never padded.
- **`PREVIEW_IMAGES`**: how many random photos Step 3 previews.

The **advanced** settings rarely need changing; each one has a comment explaining it.

In [4]:
# ======================= EDIT THESE =======================

SOURCE_DIR = None             # e.g. "My_Dataset"; None -> pick with a pop-up
OUTPUT_NAME = None            # None -> "<source name>_segmented"

PROMPT = "beetle."
BOX_THRESHOLD = 0.35          # lower -> finds more beetles (and more false ones)
TEXT_THRESHOLD = 0.25

# --- Beetle vs look-alike check (CLIP) ---
LOOKALIKES = ["bee", "wasp", "ant", "fly"]  # insects mistaken for beetles; [] -> filter off
SCORING_MODE = "joint"        # "joint": one question over every category; "separate": one yes/no question per category

# joint mode
BEETLE_MIN = 0.30             # a box needs at least this beetle score to pass (low = lenient)
LOOKALIKE_MIN = 0.85          # reject as a look-alike at this score if it also beats the beetle score (high = strict)

# separate mode
SEP_INSECT_MIN = 0.50         # "is there an insect?"  below this -> not an insect
SEP_BEETLE_MIN = 0.85         # "is it a beetle?"      at or above this -> beetle-like
SEP_LOOKALIKE_MIN = 0.50      # "is it a bee? a wasp?" best look-alike at or above this -> look-alike-like

BACKGROUND_COLOR = (255, 255, 255)
PADDING = 10                  # pixels
MAKE_SQUARE = True            # pad training JPGs to a square so Train_Model's resize doesn't stretch them

PREVIEW_IMAGES = 6
TUNING_SAMPLE_SIZE = 300      # Step 3b: random photos to score for threshold tuning

# ----------------------- advanced -------------------------

MIN_BOX_FRACTION = 0.001      # ignore boxes smaller than this fraction of the photo (dust, specks)
DUPLICATE_IOU = 0.5           # two boxes overlapping more than this are the same beetle
DUPLICATE_CONTAINED = 0.8     # ...as are two boxes where one is 80% inside the other
BOX_EXPAND = 0.05             # grow each box by 5% before tracing, so antenna and leg tips aren't cut off
HOLE_MATCH = 0.5              # fill a hole in the outline if it's at least half as different from the background as the beetle is
TINY_HOLE_FRACTION = 0.005    # always fill holes smaller than 0.5% of the beetle (glare spots)
MIN_PIECE_FRACTION = 0.01     # drop stray bits of outline smaller than 1% of the beetle
RANDOM_SEED = 42              # which photos Step 3 previews
TUNING_SEED = 7               # which photos Step 3b scores

# What CLIP compares each box with. Prompts in a group are averaged, so a few different wordings are better than one.
# Give "beetle" prompts for the beetle shapes that get confused with look-alikes (e.g. ant-like rove beetles).
BEETLE_PROMPTS = [
    "a photo of a beetle",
    "a macro photo of a beetle specimen",
    "a photo of a rove beetle",
    "a photo of a ground beetle",
    "a photo of a weevil",
    "a photo of a leaf beetle",
]
LOOKALIKE_TEMPLATES = ["a photo of a {}", "a macro photo of a {} specimen"]  # {} is filled with each LOOKALIKES name
OTHER_PROMPTS = ["a label with printed text", "a ruler or scale bar", "a piece of tape", "an empty background"]
INSECT_PROMPTS = ["a photo of an insect", "a macro photo of an insect specimen"]  # "any insect": separate mode asks "insect, or just a label?" and "beetle (or bee...), or just any insect?"

# ==========================================================


## Adding important libraries that are needed for this particular version.

In [2]:
%pip install torch
%pip install transformers

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


## Step 2: Choose the folder and load the models

Finds the folder to process (opening the picker if `SOURCE_DIR` is `None`) and loads Grounding DINO, SAM and CLIP.

What you'll see, in order:
1. **"Loading libraries..."**: importing PyTorch and transformers takes about 30 seconds in a fresh kernel. The cell is working even though nothing else appears yet.
2. **"A folder picker has opened..."**: choose the folder. The picker sometimes opens behind your other windows, so check your taskbar or press Alt+Tab if you don't see it. **The Interrupt button won't work while the picker is open.**
3. **"Loading models..."**: takes a minute the first time, while the models download, and a few seconds after that.

In [5]:
print("Loading libraries (about 30 seconds in a fresh kernel)...", flush=True)

import os
import csv
import time
import random
import shutil
from collections import Counter

import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image, ImageOps
from scipy import ndimage
from transformers import AutoModelForZeroShotObjectDetection, AutoProcessor, CLIPModel, CLIPProcessor, SamModel, SamProcessor

print("Libraries loaded.", flush=True)

DINO_MODEL = "IDEA-Research/grounding-dino-tiny"
SAM_MODEL = "facebook/sam-vit-base"
CLIP_MODEL = "openai/clip-vit-base-patch32"  # for a stronger (slower, ~1.7 GB) check: "openai/clip-vit-large-patch14"
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp"}


def find_repo_root(marker='Models'):
    path = os.path.abspath(os.getcwd())
    for _ in range(5):
        if os.path.isdir(os.path.join(path, marker)):
            return path
        parent = os.path.dirname(path)
        if parent == path:
            return os.path.abspath(os.getcwd())
        path = parent
    return os.path.abspath(os.getcwd())


def pick_folder(start_dir):
    import tkinter as tk
    from tkinter import filedialog
    root = tk.Tk()
    root.withdraw()
    # Bring the picker in front of the editor/browser. Without parent=root and
    # update(), Windows often opens it hidden behind them.
    root.attributes("-topmost", True)
    root.update()
    folder = filedialog.askdirectory(
        parent=root,
        title="Choose the folder of photos to segment",
        initialdir=start_dir)
    root.destroy()
    return folder


def overlaps(a, b):
    """True if folders a and b are the same folder or one is inside the other."""
    a, b = os.path.normcase(a), os.path.normcase(b)
    try:
        return os.path.commonpath([a, b]) in (a, b)
    except ValueError:  # different drives
        return False


REPO_ROOT = find_repo_root()
DATA_DIR = os.path.join(REPO_ROOT, "Data")

# --- Source folder ---
if SOURCE_DIR is None:
    print("A folder picker has opened. Check your taskbar if you don't see it.", flush=True)
    SOURCE_DIR = pick_folder(DATA_DIR)
    if not SOURCE_DIR:
        raise RuntimeError("No folder was chosen. Run this cell again and pick a folder.")
if not os.path.isabs(SOURCE_DIR):
    SOURCE_DIR = os.path.join(DATA_DIR, SOURCE_DIR)
SOURCE_DIR = os.path.abspath(SOURCE_DIR)
if not os.path.isdir(SOURCE_DIR):
    raise FileNotFoundError(f"This folder doesn't exist: {SOURCE_DIR}")

# --- Output folders ---
if OUTPUT_NAME is None:
    OUTPUT_NAME = os.path.basename(SOURCE_DIR) + "_segmented"
OUTPUT_DIR = os.path.join(DATA_DIR, OUTPUT_NAME)
EXTRAS_DIR = OUTPUT_DIR + "_extras"
for folder in (OUTPUT_DIR, EXTRAS_DIR):
    if overlaps(SOURCE_DIR, folder):
        raise ValueError(
            f"The source folder and {folder} can't be the same folder or inside one another. "
            "Choose a different source folder or OUTPUT_NAME.")

# --- Find every image ---
IMAGE_PATHS = []
for dirpath, dirnames, filenames in os.walk(SOURCE_DIR):
    dirnames[:] = sorted(d for d in dirnames if not d.startswith("."))
    for f in sorted(filenames):
        if os.path.splitext(f)[1].lower() in IMAGE_EXTENSIONS:
            IMAGE_PATHS.append(os.path.join(dirpath, f))
if not IMAGE_PATHS:
    raise RuntimeError(f"No images found in {SOURCE_DIR} or its subfolders.")

print(f"Source: {SOURCE_DIR}  ({len(IMAGE_PATHS)} images)")
print(f"Output: {OUTPUT_DIR}")
print(f"Extras: {EXTRAS_DIR}")

# --- Load the models ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\nLoading models on {DEVICE.upper()} (the first run downloads about 1.6 GB)...", flush=True)
dino_processor = AutoProcessor.from_pretrained(DINO_MODEL)
dino_model = AutoModelForZeroShotObjectDetection.from_pretrained(DINO_MODEL).to(DEVICE).eval()
sam_processor = SamProcessor.from_pretrained(SAM_MODEL)
sam_model = SamModel.from_pretrained(SAM_MODEL).to(DEVICE).eval()
clip_processor = CLIPProcessor.from_pretrained(CLIP_MODEL)
clip_model = CLIPModel.from_pretrained(CLIP_MODEL).to(DEVICE).eval()
print("Models loaded.")

Loading libraries (about 30 seconds in a fresh kernel)...
Libraries loaded.
A folder picker has opened. Check your taskbar if you don't see it.
Source: C:\Users\verab\Desktop\Specimen_Classifier\Data\dirty_0.95_0.5  (23 images)
Output: c:\Users\verab\Desktop\Specimen_Classifier\Data\dirty_0.95_0.5_segmented
Extras: c:\Users\verab\Desktop\Specimen_Classifier\Data\dirty_0.95_0.5_segmented_extras

Loading models on CPU (the first run downloads about 1.6 GB)...


Loading weights:   0%|          | 0/978 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/314 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Models loaded.


### The pipeline

This cell only defines the steps; nothing runs yet. You don't need to change anything here.

- **`detect_beetles`**: runs Grounding DINO, then removes boxes that are too small, boxes that wrap a whole group of beetles (Grounding DINO sometimes boxes the group as well as each beetle), and duplicate boxes around the same beetle. Then `score_box` crops each remaining box and CLIP scores it **both ways from one look at the crop**: joint (`j_` scores) and separate (`s_` scores). `decide` turns the scores of the chosen `SCORING_MODE` into **beetle**, **lookalike**, **neither**, **no_insect** or **review**. Every box is returned with its scores, so nothing is lost silently.
- **`segment_beetles`**: gives each box to SAM, which suggests 3 possible outlines. SAM's own favourite is often wrong (sometimes it's the background around the beetle), so instead the outline whose inside differs most in colour from the background is chosen. The background colour is measured along the edge of the box. Holes inside the outline are filled if they look like beetle (not background, like the gap inside a looped antenna), and stray specks are dropped.
- **`make_cutouts`**: crops the photo to each outline and builds the transparent PNG and the solid-background JPG. With `MAKE_SQUARE`, the JPG is then padded to a square, so the CNN sees the beetle's true shape.

In [6]:
def box_area(b):
    return max(0.0, b[2] - b[0]) * max(0.0, b[3] - b[1])


def box_iou(a, b):
    inter = box_area((max(a[0], b[0]), max(a[1], b[1]), min(a[2], b[2]), min(a[3], b[3])))
    union = box_area(a) + box_area(b) - inter
    return inter / union if union else 0.0


def box_containment(a, b):
    """Fraction of the smaller box that lies inside the other one."""
    inter = box_area((max(a[0], b[0]), max(a[1], b[1]), min(a[2], b[2]), min(a[3], b[3])))
    smaller = min(box_area(a), box_area(b))
    return inter / smaller if smaller else 0.0


def contains_center(outer, inner):
    cx, cy = (inner[0] + inner[2]) / 2, (inner[1] + inner[3]) / 2
    return outer[0] <= cx <= outer[2] and outer[1] <= cy <= outer[3]


@torch.no_grad()
def run_dino(image, prompt, threshold):
    """Returns [(score, [x0, y0, x1, y1]), ...] for `prompt`, most confident first, specks removed."""
    # Grounding DINO normally enlarges photos to 800 px on the short side first. That
    # triples the time for small lab photos without finding beetles any better.
    size = {"shortest_edge": min(800, image.width, image.height), "longest_edge": 1333}
    inputs = dino_processor(images=image, text=prompt, size=size, return_tensors="pt").to(DEVICE)
    outputs = dino_model(**inputs)
    result = dino_processor.post_process_grounded_object_detection(
        outputs, inputs.input_ids,
        threshold=threshold, text_threshold=TEXT_THRESHOLD,
        target_sizes=[image.size[::-1]])[0]
    detections = sorted(zip(result["scores"].tolist(), result["boxes"].tolist()), reverse=True)

    min_area = MIN_BOX_FRACTION * image.width * image.height
    return [(s, b) for s, b in detections if box_area(b) >= min_area]


import math


class Candidate:
    """One box found in a photo, with all of CLIP's scores for it and what was decided."""

    def __init__(self, score, box, scores, decision):
        self.score, self.box, self.scores, self.decision = score, box, scores, decision

    # The scores of the mode in use, so the preview and the sorting don't care which mode it is
    @property
    def beetle(self):
        return self.scores["s_beetle" if SCORING_MODE == "separate" else "j_beetle"]

    @property
    def lookalike(self):
        return self.scores["s_lookalike" if SCORING_MODE == "separate" else "j_lookalike"]

    @property
    def lookalike_name(self):
        return self.scores["s_name" if SCORING_MODE == "separate" else "j_name"]


def _tensor(x):
    """Some transformers versions return a tensor from the feature getters, others an output object."""
    return x if torch.is_tensor(x) else x.pooler_output


_TEXT_CACHE = {}


@torch.no_grad()
def class_features():
    """One CLIP text vector per class, in this order: beetle, each of LOOKALIKES, other, insect.
    Each is the average over that class's prompts. Rebuilt automatically if Step 1 changes."""
    key = (tuple(BEETLE_PROMPTS), tuple(LOOKALIKES), tuple(LOOKALIKE_TEMPLATES), tuple(OTHER_PROMPTS), tuple(INSECT_PROMPTS))
    if key not in _TEXT_CACHE:
        groups = ([BEETLE_PROMPTS] + [[t.format(name) for t in LOOKALIKE_TEMPLATES] for name in LOOKALIKES]
                  + [OTHER_PROMPTS, INSECT_PROMPTS])
        vectors = []
        for prompts in groups:
            inputs = clip_processor(text=prompts, return_tensors="pt", padding=True).to(DEVICE)
            v = _tensor(clip_model.get_text_features(**inputs))
            v = v / v.norm(dim=-1, keepdim=True)
            v = v.mean(dim=0)
            vectors.append(v / v.norm())
        _TEXT_CACHE[key] = torch.stack(vectors)
    return _TEXT_CACHE[key]


def crop_box(image, box):
    return image.crop(tuple(int(v) for v in expand_box(box, image.width, image.height)))


def _softmax(values):
    top = max(values)
    exps = [math.exp(v - top) for v in values]
    total = sum(exps)
    return [e / total for e in exps]


def _sigmoid(z):
    return 1.0 / (1.0 + math.exp(-max(-60.0, min(60.0, z))))


@torch.no_grad()
def score_box(image, box):
    """Crops the box and asks CLIP about it. Returns a dict with two sets of scores, both from the same look at the crop.

    joint (j_):    one question over [beetle, each look-alike, other]; probabilities that add up to 1.
                   j_lookalike is the highest look-alike (not their sum, which would unfairly favour a long LOOKALIKES list).
    separate (s_): one yes/no question per category, each answered on its own (they need not add up to 1):
                   s_insect:  insect, rather than a label/tape/background?
                   s_beetle:  beetle, rather than just any insect?
                   s_p[name]: that look-alike, rather than just any insect? (s_lookalike is the highest of them)"""
    inputs = clip_processor(images=crop_box(image, box), return_tensors="pt").to(DEVICE)
    v = _tensor(clip_model.get_image_features(**inputs))
    v = v / v.norm(dim=-1, keepdim=True)
    sims = (v @ class_features().T)[0].cpu().tolist()  # similarity to: beetle, each look-alike, other, insect
    scale = clip_model.logit_scale.exp().item()
    n = len(LOOKALIKES)
    beetle_sim, look_sims, other_sim, insect_sim = sims[0], sims[1:1 + n], sims[1 + n], sims[2 + n]

    joint = _softmax([scale * s for s in sims[:n + 2]])
    j_look = joint[1:1 + n]
    s_look = [_sigmoid(scale * (s - insect_sim)) for s in look_sims]
    return {
        "j_beetle": joint[0], "j_lookalike": max(j_look), "j_name": LOOKALIKES[j_look.index(max(j_look))], "j_other": joint[-1],
        "s_insect": _sigmoid(scale * (insect_sim - other_sim)),
        "s_beetle": _sigmoid(scale * (beetle_sim - insect_sim)),
        "s_lookalike": max(s_look), "s_name": LOOKALIKES[s_look.index(max(s_look))],
        "s_p": dict(zip(LOOKALIKES, s_look)),
    }


def decide_joint(beetle, lookalike, beetle_min, lookalike_min):
    """Joint rule. Returns "lookalike", "beetle" or "no_insect". (Tune_Thresholds.ipynb uses the same rule.)"""
    if lookalike >= lookalike_min and lookalike > beetle:
        return "lookalike"
    if beetle >= beetle_min:
        return "beetle"
    return "no_insect"


def decide_separate(insect, beetle, lookalike, insect_min, beetle_min, lookalike_min):
    """Separate rule. Returns "no_insect", "beetle", "review", "lookalike" or "neither"."""
    if insect < insect_min:
        return "no_insect"
    beetle_like, lookalike_like = beetle >= beetle_min, lookalike >= lookalike_min
    if beetle_like and not lookalike_like:
        return "beetle"
    if beetle_like and lookalike_like:
        return "review"
    if lookalike_like:
        return "lookalike"
    return "neither"


def decide(scores):
    """The decision for the SCORING_MODE in use."""
    if SCORING_MODE == "separate":
        return decide_separate(scores["s_insect"], scores["s_beetle"], scores["s_lookalike"],
                               SEP_INSECT_MIN, SEP_BEETLE_MIN, SEP_LOOKALIKE_MIN)
    return decide_joint(scores["j_beetle"], scores["j_lookalike"], BEETLE_MIN, LOOKALIKE_MIN)


def detect_beetles(image):
    """Returns one Candidate per box Grounding DINO found (after removing specks, group boxes and
    duplicates), top-to-bottom. Candidate.decision is "beetle", "lookalike", "neither", "no_insect" or "review"."""
    if SCORING_MODE not in ("joint", "separate"):
        raise ValueError('SCORING_MODE must be "joint" or "separate"')
    detections = run_dino(image, PROMPT, BOX_THRESHOLD)

    # A box that holds the centres of 2+ other boxes is around the group, not one beetle
    detections = [(s, b) for s, b in detections
                  if sum(contains_center(b, other) for _, other in detections if other is not b) < 2]

    kept = []
    for s, b in detections:  # most confident first, so duplicates lose to the better box
        if all(box_iou(b, k) < DUPLICATE_IOU and box_containment(b, k) < DUPLICATE_CONTAINED for _, k in kept):
            kept.append((s, b))

    candidates = []
    for s, b in kept:
        if LOOKALIKES:
            scores = score_box(image, b)
            candidates.append(Candidate(s, b, scores, decide(scores)))
        else:  # filter off: every box is a beetle
            scores = {"j_beetle": 1.0, "j_lookalike": 0.0, "j_name": "", "j_other": 0.0,
                      "s_insect": 1.0, "s_beetle": 1.0, "s_lookalike": 0.0, "s_name": "", "s_p": {}}
            candidates.append(Candidate(s, b, scores, "beetle"))
    candidates.sort(key=lambda c: (c.box[1], c.box[0]))
    return candidates


def expand_box(b, width, height):
    dx, dy = (b[2] - b[0]) * BOX_EXPAND, (b[3] - b[1]) * BOX_EXPAND
    return [max(0.0, b[0] - dx), max(0.0, b[1] - dy), min(width, b[2] + dx), min(height, b[3] + dy)]


def box_slice(box, height, width):
    x0, y0 = max(0, int(box[0])), max(0, int(box[1]))
    x1, y1 = min(width, int(np.ceil(box[2]))), min(height, int(np.ceil(box[3])))
    return np.s_[y0:y1, x0:x1]


def background_distance(pixels, box):
    """How different each pixel's colour is from the background. The background colour is
    taken from the box's border, which (after expand_box) is almost never on the beetle."""
    inside = pixels[box_slice(box, *pixels.shape[:2])]
    border = np.concatenate([inside[0], inside[-1], inside[:, 0], inside[:, -1]]).astype(np.float32)
    return np.linalg.norm(pixels.astype(np.float32) - np.median(border, axis=0), axis=2)


def choose_mask(candidates, distance, box):
    """Pick the SAM candidate that stands out most from the background inside the box.
    SAM's own confidence often prefers a patch of background, so it isn't used here."""
    region = box_slice(box, *distance.shape)
    best, best_score = None, 0.0  # a candidate must at least differ more inside than outside
    for mask in candidates:
        m = mask[region]
        if m.all() or not m.any():
            continue
        score = distance[region][m].mean() - distance[region][~m].mean()
        if score > best_score:
            best, best_score = mask, score
    return best


def clean_mask(mask, distance):
    # Fill holes inside the outline, but only ones that look like beetle (e.g. a body SAM
    # traced around) or are tiny (glare), not background (e.g. the gap inside a looped antenna)
    filled = ndimage.binary_fill_holes(mask)
    holes, n = ndimage.label(filled & ~mask)
    if n:
        ids = np.arange(1, n + 1)
        hole_sizes = ndimage.sum(np.ones_like(mask), holes, ids)
        hole_colour = ndimage.mean(distance, holes, ids)
        fill = (hole_colour >= HOLE_MATCH * distance[mask].mean()) | (hole_sizes <= TINY_HOLE_FRACTION * mask.sum())
        mask = mask | np.isin(holes, ids[fill])

    labels, n = ndimage.label(mask)
    if n > 1:
        sizes = ndimage.sum(mask, labels, range(1, n + 1))
        keep = np.flatnonzero(sizes >= MIN_PIECE_FRACTION * sizes.max()) + 1
        mask = np.isin(labels, keep)
    return mask


@torch.no_grad()
def segment_beetles(image, boxes):
    """Returns one True/False mask (height x width) per box, or None where no outline stood out."""
    boxes = [expand_box(b, image.width, image.height) for b in boxes]
    inputs = sam_processor(images=image, input_boxes=[boxes], return_tensors="pt").to(DEVICE)
    outputs = sam_model(**inputs, multimask_output=True)  # 3 candidate outlines per box
    candidates = sam_processor.image_processor.post_process_masks(
        outputs.pred_masks.cpu(), inputs["original_sizes"].cpu(), inputs["reshaped_input_sizes"].cpu())[0]
    pixels = np.asarray(image)
    masks = []
    for box, box_candidates in zip(boxes, candidates):
        distance = background_distance(pixels, box)
        mask = choose_mask([c.numpy() for c in box_candidates], distance, box)
        masks.append(None if mask is None else clean_mask(mask, distance))
    return masks


def pad_to_square(image):
    """Centre the image on a square of BACKGROUND_COLOR, at its original resolution.
    Keeps the beetle's proportions, so Train_Model's resize to 224x224 doesn't stretch it."""
    side = max(image.size)
    square = Image.new("RGB", (side, side), BACKGROUND_COLOR)
    square.paste(image, ((side - image.width) // 2, (side - image.height) // 2))
    return square


def make_cutouts(image, mask):
    """Returns (transparent PNG, training image), cropped to the mask. None if the mask is empty."""
    ys, xs = np.nonzero(mask)
    if len(xs) == 0:
        return None
    x0, y0 = max(0, xs.min() - PADDING), max(0, ys.min() - PADDING)
    x1, y1 = min(image.width, xs.max() + 1 + PADDING), min(image.height, ys.max() + 1 + PADDING)
    rgb = np.asarray(image)[y0:y1, x0:x1]
    alpha = mask[y0:y1, x0:x1].astype(np.uint8) * 255
    png = Image.fromarray(np.dstack([rgb, alpha]), "RGBA")
    solid = Image.new("RGB", png.size, BACKGROUND_COLOR)
    solid.paste(png, mask=png.getchannel("A"))
    if MAKE_SQUARE:
        solid = pad_to_square(solid)
    return png, solid


def process_image(path):
    """Returns (image, beetles, rejected, reviewed).
    beetles:  [(candidate, png, solid), ...], one per box that passed.
    rejected: [candidate, ...], boxes decided to be a look-alike, neither, or not an insect.
    reviewed: [(candidate, png, solid), ...], boxes set aside for review (separate mode only); they get cutouts
              too, so confirmed beetles can go straight into the dataset."""
    image = ImageOps.exif_transpose(Image.open(path)).convert("RGB")
    candidates = detect_beetles(image)
    to_segment = [c for c in candidates if c.decision in ("beetle", "review")]
    rejected = [c for c in candidates if c.decision not in ("beetle", "review")]
    if not to_segment:
        return image, [], rejected, []
    masks = segment_beetles(image, [c.box for c in to_segment])
    beetles, reviewed = [], []
    for c, mask in zip(to_segment, masks):
        cutouts = None if mask is None else make_cutouts(image, mask)
        if cutouts is not None:
            (beetles if c.decision == "beetle" else reviewed).append((c, *cutouts))
        elif c.decision == "review":
            rejected.append(c)  # no outline found: keep it as a plain crop so it isn't lost
    return image, beetles, rejected, reviewed


## Step 3: Preview

Runs the pipeline on a few random photos and shows, for each one, the photo with the boxes Grounding DINO found (and how confident it was) next to each beetle's **training image**: exactly what Step 4 saves for `Train_Model.ipynb`, padded to a square when `MAKE_SQUARE` is on. **Nothing is saved.**

Each box is labelled with its scores: **B** is the beetle score and **L** the highest look-alike score (and which look-alike); in separate mode **I** is the insect score as well. Kept beetles are **red**. Rejected boxes are dashed: **blue** for a look-alike, **orange** for neither a beetle nor a look-alike, **grey** for not an insect. **Purple** boxes need review (separate mode: scored as both a beetle and a look-alike). Seeing the scores on real beetles shows how close they came to being rejected.

Check that:
- every beetle has a box, and nothing that isn't a beetle does. If not, adjust `BOX_THRESHOLD` in Step 1.
- bees (or other look-alikes) get a dashed blue box, and no real beetle does. If real beetles are rejected, loosen the beetle threshold or tighten the look-alike one; if look-alikes slip through, do the opposite. For more than a few photos, use Step 3b and `Tune_Thresholds.ipynb` instead of adjusting by eye.
- the outlines keep the legs and antennae. If tips are cut off, raise `BOX_EXPAND`.
- the beetles keep their true shape (long beetles stay long) inside the square.

After changing a setting, re-run Step 1 and this cell; you don't need to re-run Step 2. Re-run just this cell to see different photos, or set `RANDOM_SEED` to another number.

In [7]:
MAX_CUTOUTS_SHOWN = 6
REJECT_COLOURS = {"lookalike": "blue", "neither": "darkorange", "no_insect": "gray", "review": "purple"}
REJECT_WORDS = {"neither": "neither beetle nor look-alike", "no_insect": "not an insect", "review": "NEEDS REVIEW"}

sample = random.Random(RANDOM_SEED).sample(IMAGE_PATHS, min(PREVIEW_IMAGES, len(IMAGE_PATHS)))
RANDOM_SEED += 1  # a different sample next time this cell runs


def score_text(c):
    insect = f"I {c.scores['s_insect']:.2f}  " if SCORING_MODE == "separate" else ""
    return f"{insect}B {c.beetle:.2f}  L {c.lookalike:.2f} {c.lookalike_name}"


for path in sample:
    start = time.time()
    image, beetles, rejected, reviewed = process_image(path)
    seconds = time.time() - start

    shown = beetles[:MAX_CUTOUTS_SHOWN]
    fig, axes = plt.subplots(1, 1 + max(len(shown), 1), figsize=(4 * (1 + max(len(shown), 1)), 4),
                             gridspec_kw={"width_ratios": [2] + [1] * max(len(shown), 1)}, squeeze=False)
    axes = axes[0]
    axes[0].imshow(image)
    for n, (c, _, _) in enumerate(beetles, 1):
        box = c.box
        axes[0].add_patch(Rectangle((box[0], box[1]), box[2] - box[0], box[3] - box[1],
                                    fill=False, edgecolor="red", linewidth=1.5))
        axes[0].text(box[0], box[1] - 3, f"{n}: {score_text(c)}", color="red", fontsize=8,
                     bbox={"facecolor": "white", "alpha": 0.7, "pad": 1, "edgecolor": "none"})
    for c in rejected + [r[0] for r in reviewed]:
        box, colour = c.box, REJECT_COLOURS[c.decision]
        what = f"look-alike ({c.lookalike_name})" if c.decision == "lookalike" else REJECT_WORDS[c.decision]
        axes[0].add_patch(Rectangle((box[0], box[1]), box[2] - box[0], box[3] - box[1],
                                    fill=False, edgecolor=colour, linewidth=1.5, linestyle="--"))
        axes[0].text(box[0], box[3] + 12, f"{what}: {score_text(c)}", color=colour, fontsize=8,
                     bbox={"facecolor": "white", "alpha": 0.7, "pad": 1, "edgecolor": "none"})
    found = f"{len(beetles)} beetle(s)" if beetles else "NO BEETLE: would go to no_beetle/"
    if reviewed:
        found += f", {len(reviewed)} need review"
    if len(rejected):
        found += f", {len(rejected)} rejected"
    axes[0].set_title(f"{os.path.relpath(path, SOURCE_DIR)}\n{found}  ({seconds:.1f} s)", fontsize=8)

    for n, ax in enumerate(axes[1:], 1):
        if n <= len(shown):
            training_image = shown[n - 1][2]  # exactly what gets saved for Train_Model
            ax.imshow(training_image)
            ax.set_title(f"beetle {n}  ({training_image.width}x{training_image.height})", fontsize=8)
            ax.set_xticks([])
            ax.set_yticks([])
            for spine in ax.spines.values():
                spine.set_edgecolor("#9a9a9a")  # frame, so the white padding's edge is visible
        else:
            ax.axis("off")
    axes[0].axis("off")
    if len(beetles) > MAX_CUTOUTS_SHOWN:
        fig.suptitle(f"(showing the first {MAX_CUTOUTS_SHOWN} of {len(beetles)} cutouts)", fontsize=8)
    plt.tight_layout()
    plt.show()


: 

## Step 3b: Score a sample to tune the thresholds

Choosing thresholds by eye on a few photos is slow and noisy. This step scores a random sample of `TUNING_SAMPLE_SIZE` photos **once**, and saves every box with its scores, so that `Tune_Thresholds.ipynb` can try hundreds of threshold combinations instantly, for **both** scoring modes, without running any model again. (CLIP's joint and separate scores both come from the same look at each crop, so scoring both costs almost nothing extra.)

It runs Grounding DINO and CLIP only (no SAM), so it is quicker than Step 4. For each photo it saves, in `Data/<output>_extras/tuning/`:

- `candidates/c00001.jpg, ...`: a small crop of every box found. The file names don't reveal the scores, so labelling isn't biased.
- `candidate_scores.csv`: for each crop, the photo it came from and its scores: `j_` columns for joint mode, `s_` columns for separate mode (including one `s_p_<name>` column per look-alike).
- `scoring_config.json`: the settings the scores depend on.

**You can stop and resume.** If you change `LOOKALIKES`, the prompt lists, `CLIP_MODEL`, `PROMPT` or `BOX_THRESHOLD`, this step refuses to continue until you delete the `tuning` folder, since the saved scores would no longer match. Changing only the thresholds (`BEETLE_MIN`, `SEP_BEETLE_MIN`, etc.) does *not* need a re-run.

Include some photos you know contain bees or wasps in the sample (e.g. by choosing a source folder that has them). Because the tuning notebook measures how often each *true* kind of box is handled correctly, it is fine to over-represent look-alikes as long as you choose them by what they are, not by what the model scored.


In [ ]:
import json

if not LOOKALIKES:
    raise ValueError("LOOKALIKES is empty, so there is nothing to score. Add some look-alikes in Step 1.")

TUNING_DIR = os.path.join(EXTRAS_DIR, "tuning")
CROP_DIR = os.path.join(TUNING_DIR, "candidates")
SCORES_PATH = os.path.join(TUNING_DIR, "candidate_scores.csv")
CONFIG_PATH = os.path.join(TUNING_DIR, "scoring_config.json")
DONE_PATH = os.path.join(TUNING_DIR, "scored_photos.txt")  # photos finished, including ones with no boxes
os.makedirs(CROP_DIR, exist_ok=True)

SCORE_COLUMNS = (["id", "source", "box", "dino_score", "j_beetle", "j_lookalike", "j_name", "j_other",
                  "s_insect", "s_beetle", "s_lookalike", "s_name"] + [f"s_p_{name}" for name in LOOKALIKES])
config = {"lookalikes": list(LOOKALIKES), "beetle_prompts": list(BEETLE_PROMPTS), "lookalike_templates": list(LOOKALIKE_TEMPLATES),
          "other_prompts": list(OTHER_PROMPTS), "insect_prompts": list(INSECT_PROMPTS),
          "clip_model": CLIP_MODEL, "prompt": PROMPT, "box_threshold": BOX_THRESHOLD}
RESET_MESSAGE = (f"The settings that the scores depend on (LOOKALIKES, the prompt lists, CLIP_MODEL, PROMPT, BOX_THRESHOLD) have changed, or the folder was made "
                 f"by an older version. Delete this folder and run this cell again:\n    {TUNING_DIR}")
if os.path.exists(CONFIG_PATH):
    with open(CONFIG_PATH, encoding="utf-8") as fh:
        if json.load(fh) != config:
            raise ValueError(RESET_MESSAGE)
elif os.path.exists(SCORES_PATH) and os.path.getsize(SCORES_PATH) > 0:
    raise ValueError(RESET_MESSAGE)  # scores from before the config file existed
else:
    with open(CONFIG_PATH, "w", encoding="utf-8") as fh:
        json.dump(config, fh, indent=1)

new_scores = not os.path.exists(SCORES_PATH) or os.path.getsize(SCORES_PATH) == 0
next_id = 1
if not new_scores:
    with open(SCORES_PATH, newline="", encoding="utf-8") as fh:
        reader = csv.reader(fh)
        if next(reader) != SCORE_COLUMNS:
            raise ValueError(RESET_MESSAGE)
        for row in reader:
            next_id = max(next_id, int(row[0][1:]) + 1)
scored_photos = set()
if os.path.exists(DONE_PATH):
    with open(DONE_PATH, encoding="utf-8") as fh:
        scored_photos = {line.rstrip("\n") for line in fh}


def score_row(cid, rel_path, c):
    s = c.scores
    return ([cid, rel_path, ";".join(f"{v:.1f}" for v in c.box), f"{c.score:.4f}",
             f"{s['j_beetle']:.4f}", f"{s['j_lookalike']:.4f}", s["j_name"], f"{s['j_other']:.4f}",
             f"{s['s_insect']:.4f}", f"{s['s_beetle']:.4f}", f"{s['s_lookalike']:.4f}", s["s_name"]]
            + [f"{s['s_p'][name]:.4f}" for name in LOOKALIKES])


order = list(IMAGE_PATHS)
random.Random(TUNING_SEED).shuffle(order)  # a bigger TUNING_SAMPLE_SIZE extends the same sample
sample = order[:min(TUNING_SAMPLE_SIZE, len(order))]
todo = [p for p in sample if os.path.relpath(p, SOURCE_DIR) not in scored_photos]
print(f"{len(sample)} photos in the sample: {len(sample) - len(todo)} already scored, {len(todo)} to go.\n")

start = time.time()
with open(SCORES_PATH, "a", newline="", encoding="utf-8") as fh, open(DONE_PATH, "a", encoding="utf-8") as done_fh:
    writer = csv.writer(fh)
    if new_scores:
        writer.writerow(SCORE_COLUMNS)
    for i, path in enumerate(todo, 1):
        rel_path = os.path.relpath(path, SOURCE_DIR)
        try:
            image = ImageOps.exif_transpose(Image.open(path)).convert("RGB")
            candidates = detect_beetles(image)
        except Exception as e:
            print(f"ERROR on {rel_path}: {e}")
            continue
        for c in candidates:
            cid = f"c{next_id:05d}"
            next_id += 1
            crop_box(image, c.box).save(os.path.join(CROP_DIR, cid + ".jpg"), quality=95)
            writer.writerow(score_row(cid, rel_path, c))
        fh.flush()
        done_fh.write(rel_path + "\n")
        done_fh.flush()
        if i % 10 == 0 or i == len(todo):
            elapsed = time.time() - start
            print(f"{i}/{len(todo)} photos  |  {elapsed / 60:.1f} min so far, about {elapsed / i * (len(todo) - i) / 60:.1f} min left")

print(f"\nDone. Scores and crops are in {TUNING_DIR}")
print("Next: open Tune_Thresholds.ipynb and point it at that folder.")


## Step 4: Process every photo

Runs the pipeline on every photo and saves the results (see the folder layout at the top). Progress and an estimated time remaining are printed every 10 photos.

**You can stop and resume.** Every finished photo is recorded in `segmentation_log.csv` straight away, so if you interrupt this cell (or close VS Code), running it again carries on where it stopped. Rejected boxes are saved as small crops in `rejected/lookalike/<name>/`, `rejected/neither/` and `rejected/no_insect/` (same folder layout as the source) so you can check them. In separate mode, boxes that scored as both a beetle and a look-alike are saved in `review/` with a training-ready JPG and a PNG cutout each: check them, and copy the real beetles into the matching folder of the output. A photo where *no* box passed counts as "no beetle" and is also copied to `no_beetle/`. The log's `rejected` and `review` columns count those boxes for each photo. If you have a log from an older version of this notebook, delete the two output folders first. To redo everything from scratch, for example after changing the thresholds, delete the two output folders first or set a new `OUTPUT_NAME`.

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(EXTRAS_DIR, exist_ok=True)
log_path = os.path.join(EXTRAS_DIR, "segmentation_log.csv")
LOG_COLUMNS = ["source", "status", "beetles", "scores", "outputs", "rejected", "review"]


def reject_subdir(c):
    """Where (inside EXTRAS_DIR) a box that didn't pass is saved."""
    if c.decision == "lookalike":
        return os.path.join("rejected", "lookalike", c.lookalike_name)
    if c.decision == "review":
        return "review"
    return os.path.join("rejected", c.decision)


new_log = not os.path.exists(log_path) or os.path.getsize(log_path) == 0
done = set()
if not new_log:
    with open(log_path, newline="", encoding="utf-8") as fh:
        reader = csv.DictReader(fh)
        if reader.fieldnames != LOG_COLUMNS:
            raise ValueError("segmentation_log.csv was made by an older version of this notebook (different columns). "
                             "Delete the two output folders, or set a new OUTPUT_NAME, and run this cell again.")
        # Photos that errored last time are tried again
        done = {row["source"] for row in reader if not row["status"].startswith("error")}

todo = [p for p in IMAGE_PATHS if os.path.relpath(p, SOURCE_DIR) not in done]
print(f"{len(IMAGE_PATHS)} photos: {len(done)} already done, {len(todo)} to go.\n")

counts = Counter()
start = time.time()
with open(log_path, "a", newline="", encoding="utf-8") as fh:
    writer = csv.writer(fh)
    if new_log:
        writer.writerow(LOG_COLUMNS)

    for i, path in enumerate(todo, 1):
        rel_path = os.path.relpath(path, SOURCE_DIR)
        rel_dir, filename = os.path.split(rel_path)
        stem = os.path.splitext(filename)[0]

        try:
            image, beetles, rejected, reviewed = process_image(path)
        except Exception as e:
            writer.writerow([rel_path, f"error: {e}", 0, "", "", 0, 0])
            counts["error"] += 1
            print(f"ERROR on {rel_path}: {e}")
            continue

        n_rejected = sum(c.decision != "review" for c in rejected)
        n_review = len(reviewed) + len(rejected) - n_rejected

        for n, c in enumerate(rejected, 1):
            reject_dir = os.path.join(EXTRAS_DIR, reject_subdir(c), rel_dir)
            os.makedirs(reject_dir, exist_ok=True)
            crop_box(image, c.box).save(os.path.join(reject_dir, f"{stem}_reject{n:02d}.jpg"), quality=95)
            counts["rejected_" + c.decision] += 1
        if reviewed:
            review_dir = os.path.join(EXTRAS_DIR, "review", rel_dir)
            os.makedirs(review_dir, exist_ok=True)
            for n, (c, png, solid) in enumerate(reviewed, 1):
                name = f"{stem}_review{n:02d}"
                solid.save(os.path.join(review_dir, name + ".jpg"), quality=95)
                png.save(os.path.join(review_dir, name + ".png"))
                counts["review"] += 1

        if not beetles:
            dest_dir = os.path.join(EXTRAS_DIR, "no_beetle", rel_dir)
            os.makedirs(dest_dir, exist_ok=True)
            shutil.copy2(path, os.path.join(dest_dir, filename))
            writer.writerow([rel_path, "no_beetle", 0, "", "", n_rejected, n_review])
            counts["no_beetle"] += 1
        else:
            jpg_dir = os.path.join(OUTPUT_DIR, rel_dir)
            png_dir = os.path.join(EXTRAS_DIR, "cutouts", rel_dir)
            os.makedirs(jpg_dir, exist_ok=True)
            os.makedirs(png_dir, exist_ok=True)
            names = []
            for n, (c, png, solid) in enumerate(beetles, 1):
                name = f"{stem}_beetle{n:02d}" if len(beetles) > 1 else stem
                solid.save(os.path.join(jpg_dir, name + ".jpg"), quality=95)
                png.save(os.path.join(png_dir, name + ".png"))
                names.append(name)
            writer.writerow([rel_path, "ok", len(beetles),
                             ";".join(f"{c.score:.2f}" for c, *_ in beetles), ";".join(names), n_rejected, n_review])
            counts["ok"] += 1
            counts["beetles"] += len(beetles)
        fh.flush()

        if i % 10 == 0 or i == len(todo):
            elapsed = time.time() - start
            remaining = elapsed / i * (len(todo) - i)
            print(f"{i}/{len(todo)} photos  |  {elapsed / 60:.1f} min so far, about {remaining / 60:.1f} min left")

print(f"\nDone. {counts['ok']} photos gave {counts['beetles']} beetle cutouts.")
for decision, what in (("lookalike", "as look-alikes"), ("neither", "as neither a beetle nor a look-alike"), ("no_insect", "as not an insect")):
    if counts["rejected_" + decision]:
        print(f"{counts['rejected_' + decision]} boxes were rejected {what}; crops are in {os.path.join(EXTRAS_DIR, 'rejected', decision)}.")
n_review_total = counts["review"] + counts["rejected_review"]
if n_review_total:
    print(f"{n_review_total} boxes need review (they scored as both a beetle and a look-alike): see {os.path.join(EXTRAS_DIR, 'review')}. "
          "The JPGs there are training-ready; copy the real beetles into the matching folder of the output.")
if counts["no_beetle"]:
    print(f"{counts['no_beetle']} photos had no beetle; copies are in {os.path.join(EXTRAS_DIR, 'no_beetle')} for review.")
if counts["error"]:
    print(f"{counts['error']} photos couldn't be processed; see the 'status' column of {log_path}.")
print(f"\nTo train on the cutouts, set this line in Train_Model.ipynb:")
print(f'    dataset_dir = os.path.join(REPO_ROOT, "Data", "{OUTPUT_NAME}")')
